# Setup

In [1]:
# base
import os
import sys
import gc
import re
import warnings
import logging
import pickle
from time import ctime, time
from datetime import timedelta
from collections import Counter

# data manipulation
import numpy as np
import pandas as pd
import itables
import seaborn as sns
import matplotlib.pyplot as plt
from rpy2.robjects.conversion import localconverter

# single cell
import anndata as ad
import scanpy as sc

# import liana as li
# import celltypist

# custom
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from utils import *

itables.init_notebook_mode(connected=True)  # Use connected=False for offline use
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", pd.errors.DtypeWarning)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.WARNING)

CORES = 10
DATADIR = Path("../../../data")
REFDIR = Path("../../../references")
MAIN_DIR = DATADIR / "processed" / "single_cell" / "combined"
SUBSETS_DIR = DATADIR / "processed" / "single_cell" / "combined" / "subsets"

METADATA = [
    "Diet",
    "Age",
    "Depot",
    "Sex",
]
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]
INT_KEY = "INT_harmony-Identifier"

converter = get_converter()
%load_ext rpy2.ipython
%matplotlib inline
# R_preload()
mpl.rcdefaults()
gc.collect()

/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.


361

# Cleaning

In [45]:
annotation = "All_Data.h5ad"
adata = sc.read_h5ad(DATADIR / "processed" / "single_cell" / "combined" / annotation)
adata.X = adata.layers["normalized"].copy()

gc.collect()
adata

AnnData object with n_obs × n_vars = 288258 × 29069
    obs: 'Identifier', 'n_genes', 'Dataset', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'Diet', 'Age', 'Sex', 'Depot', 'celltype_So2025', 'celltypist_So2025', 'celltype_Emont2022', 'celltypist_Emont2022', 'celltype'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'PCA', 'PCA_hvg', 'UMAP_INT_harmony-Dataset', 'UMAP_INT_harmony-Identifier', 'UMAP_INT_harmony_hvg-Dataset', 'UMAP_INT_harmony_hvg-Identifier', 'UMAP_INT_none', 'UMAP_INT_none_hvg', 'UMAP_INT_scanorama-Dataset', 'UMAP_INT_scanorama-Identifier', 'UMAP_INT_scanorama_hvg-Dataset', 'UMAP_INT_scanorama_hvg-Identifier', 'hvg', 'log1p', 'methods', 'neighbors', 'neighbors_INT_harmony-Dataset', 'neighbors_INT_harmony-Iden

In [ ]:
f = plt.figure(figsize=(18, 12), layout="constrained")
check_integration(
    adata,
    "Dataset",
    f,
    embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
    nrow=2,
    ncol=2,
)

for col in METADATA[:-1]:
    f = plt.figure(figsize=(18, 9), layout="constrained")
    check_integration(
        adata,
        col,
        f,
        embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
        nrow=1,
        ncol=2,
    )

f = plt.figure(figsize=(18, 15), layout="constrained")
check_integration(
    adata,
    "Age",
    f,
    embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
    nrow=3,
    ncol=2,
)

Filters:
* remove female mice
* remove 8 & 88 weeks
* remove iWAT

In [8]:
adata = adata[~(adata.obs["Sex"].isin(["Female"]))]
adata = adata[~(adata.obs["Age"].isin(["8 weeks", "88 weeks"]))]
adata = adata[~(adata.obs["Depot"].isin(["iWAT"]))]

In [ ]:
f = plt.figure(figsize=(18, 12), layout="constrained")
check_integration(
    adata,
    "Dataset",
    f,
    embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
    nrow=2,
    ncol=2,
)

f = plt.figure(figsize=(18, 9), layout="constrained")
check_integration(
    adata,
    "Diet",
    f,
    embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
    nrow=1,
    ncol=2,
)

f = plt.figure(figsize=(18, 15), layout="constrained")
check_integration(
    adata,
    "Age",
    f,
    embeddings=[f"UMAP_{INT_KEY}", f"LocalMAP_{INT_KEY}"],
    nrow=3,
    ncol=2,
)

# Adipocytes

### Setup

In [ ]:
CELLTYPE_ADDON = "adipo"
CLUSTER_KEY = "leiden_adipo"
DE_KEY = "adipo_DEGs"
adata.obs["celltype"].cat.categories.tolist()

In [ ]:
# subset
adata_adipo = adata[adata.obs["celltype"] == "adipocyte"].copy()
del adata_adipo.uns, adata_adipo.varm, adata_adipo.obsp
sc.pp.filter_cells(adata_adipo, min_genes=100)
sc.pp.filter_genes(adata_adipo, min_cells=5)
Visualize(adata_adipo, CELLTYPE_ADDON, input_key="INT_harmony-Identifier")

In [ ]:
# metadata checks
f = plt.figure(figsize=(18, 12), layout="constrained")
check_integration(
    adata_adipo,
    "Dataset",
    f,
    embeddings=[f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"],
    nrow=2,
    ncol=2,
)

f = plt.figure(figsize=(18, 9), layout="constrained")
check_integration(
    adata_adipo,
    "Diet",
    f,
    embeddings=[f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"],
    nrow=1,
    ncol=2,
)

f = plt.figure(figsize=(18, 15), layout="constrained")
check_integration(
    adata_adipo,
    "Age",
    f,
    embeddings=[f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"],
    nrow=2,
    ncol=2,
)

for col in ["Depot", "Sex"]:
    f = plt.figure(figsize=(14, 6), layout="constrained")
    check_integration(
        adata_adipo,
        col,
        f,
        embeddings=[f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"],
        mini=True,
    )

In [ ]:
# cluster
cluster = True
resolutions = np.arange(1, 13) / 10

# cluster
if cluster is True:
    Cluster(adata_adipo, CLUSTER_KEY, resolutions, neighbor_key="neighbors_adipo")

# plot clusters
for embedding in [f"UMAP_{CELLTYPE_ADDON}", f"LocalMAP_{CELLTYPE_ADDON}"]:
    r, c = 4, 3
    f, axs = plt.subplots(r, c, figsize=(8 * c, 6 * r), layout="constrained")
    axs = axs.flatten()

    for i, res in enumerate(resolutions):
        res_key = f"{CLUSTER_KEY}_{res}"
        cluster_c = color_gen(adata_adipo.obs[res_key])
        adata_adipo.obs[res_key] = (
            adata_adipo.obs[res_key].astype(int).astype("category")
        )

        sc.pl.embedding(
            adata_adipo,
            basis=embedding,
            color=res_key,
            ax=axs[i],
            show=False,
            legend_loc="on data",
            legend_fontoutline=2,
            legend_fontsize=15,
            palette=cluster_c,
        )
        axs[i].annotate(
            f"n = {adata_adipo.shape[0]}",
            size=15,
            fontweight="bold",
            xy=(0.98, 0.02),
            xycoords="axes fraction",
            horizontalalignment="right",
            verticalalignment="bottom",
        )

### Check resolutions

In [ ]:
embedding = f"UMAP_{CELLTYPE_ADDON}"
for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res}"
    cluster_c = color_gen(adata_adipo.obs[res_key])
    adata_adipo.obs[res_key] = adata_adipo.obs[res_key].astype(int).astype("category")

    f, ax = plt.subplots(1, 1, figsize=(20, 16))
    sc.pl.embedding(
        adata_adipo,
        basis=embedding,
        color=res_key,
        ax=ax,
        show=False,
        legend_loc="on data",
        legend_fontoutline=2,
        legend_fontsize=30,
        palette=cluster_c,
        size=20,
    )
    ax.annotate(
        f"n = {adata_adipo.shape[0]}",
        size=15,
        fontweight="bold",
        xy=(0.98, 0.02),
        xycoords="axes fraction",
        horizontalalignment="right",
        verticalalignment="bottom",
    )

In [ ]:
# barplot counts
resolutions = np.arange(1, 13) / 10

r, c = 3, 4
f, axs = plt.subplots(r, c, figsize=(10 * c, 5 * r), layout="constrained")
axs = axs.flatten()
for i, res in enumerate(resolutions):
    res_key = f"{CLUSTER_KEY}_{res}"

    plt.figure()
    pd.Series(Counter(adata_adipo.obs[res_key])).sort_index().plot(
        kind="bar", color=adata_adipo.uns[f"{res_key}_colors"], rot=30, ax=axs[i]
    )
    axs[i].bar_label(axs[i].containers[0])
    axs[i].set_title(f"{res_key} counts")

In [ ]:
# percentage breakdowns

for col in ["Dataset"] + METADATA[:2]:
    r, c = 3, 4
    f, axs = plt.subplots(r, c, figsize=(10 * c, 6 * r), layout="constrained")
    axs = axs.flatten()
    for i, res in enumerate(resolutions):
        res_key = f"{CLUSTER_KEY}_{res}"
        plot_cluster_stackedbarplot(adata_adipo, res_key, col, pct=True, ax=axs[i])
    f.suptitle(col + " Split", size=30)

# plot_cluster_riverplot(adata_adipo.obs[f"{CLUSTER_KEY}_0.5"], adata_adipo.obs[f"{CLUSTER_KEY}_0.6"])

### Markers

In [ ]:
adp_markers = {
    "Dividing genes": ["Mki67", "Top2a", "Ube2c"],
}

unique_adp_markers = pd.Series(list(set([i for j in fb_markers.values() for i in j])))
unique_adp_markers[~unique_adp_markers.isin(adata_adipo.var_names)].tolist()

In [ ]:
for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    plot_violinplot(adata_adipo, res_key, fb_markers, useStripPlot=False)

In [ ]:
plot_cluster_violinplot(adata_adipo, "Dataset", res_key, mac_markers)

### DEGs

In [ ]:
RUN_DEG = True

# change to string for DEGs
for col in adata_adipo.obs.columns:
    if CLUSTER_KEY in col:
        adata_adipo.obs[col] = adata_adipo.obs[col].astype(str).astype("category")

# calculate DEGs
if RUN_DEG is True:
    clear_uns(adata_adipo, res_key)
    sc.tl.rank_genes_groups(
        adata_adipo,
        groupby=res_key,
        key_added=de_key,
        use_raw=False,
        layer="normalized",
        method="wilcoxon",
    )

In [ ]:
for res in SELECT_RES:
    res_key = f"{CLUSTER_KEY}_{res:.1f}"
    de_key = f"{DE_KEY}_{res:.1f}"
    n_clust = len(adata_adipo.obs[res_key].unique())

    # plot DEGs
    top_genes = {}
    for group, df in sc.get.rank_genes_groups_df(
        adata_adipo, group=None, key=de_key
    ).groupby("group"):
        top_genes[group] = df["names"][:30].tolist()
    f, ax = plt.subplots(1, 1, figsize=(n_clust * 5, 5), layout="constrained")
    sc.pl.rank_genes_groups_dotplot(
        adata_adipo,
        groupby=res_key,
        key=de_key,
        # n_genes=20,
        var_names=top_genes,
        # standard_scale="var",
        values_to_plot="logfoldchanges",
        cmap="bwr",
        colorbar_title="log fold change",
        ax=ax,
        title=f"{res_key}_DEGs",
        vmin=-4,
        vmax=4,
    )

    sc.pl.rank_genes_groups_heatmap(
        adata_adipo,
        key=de_key,
        groupby=res_key,
        layer="normalized",
        n_genes=50,
    )
    sc.pl.rank_genes_groups(
        adata_adipo, key=de_key, n_genes=30, title=f"{res_key}_DEGs"
    )

### Specific comparisons

## Save/Load

In [ ]:
# save
clear_uns(adata_adipo, "dendrogram")
clear_uns(adata_adipo, "colors")
CELLTYPE_ADDON = "adipo"
adata_adipo.write_zarr(SUBSETS_DIR / CELLTYPE_ADDON)

In [ ]:
# load
CELLTYPE_ADDON = "adipo"
adata_adipo = ad.read_zarr(SUBSETS_DIR / CELLTYPE_ADDON)
adata_adipo